In [1]:
# Import libraries

import pandas as pd
import plotly.express as px
from dash import html, dcc, Output, Input, callback, Dash
import dash_bootstrap_components as dbc
import openpyxl
import webbrowser

In [5]:
    # Read files

    population = pd.read_excel(r"C:\Users\Ulvi\Downloads\Population.xlsx")
    demographics = pd.read_excel(r"C:\Users\Ulvi\Downloads\Population by Age.xlsx")
    macroeconomics = pd.read_excel(r"C:\Users\Ulvi\Downloads\Macroeconomic.xlsx")

    #Organize dataframe
    macroeconomics= macroeconomics.rename(columns = {'Budget revenue (bln AZN)':'Budget revenue', 
                                                     'Budget expenditure (bln AZN)':'Budget expenditure'})

    # Combine two tables for dropdown
    years = sorted(set(population['Year']).union(macroeconomics['Year']))

    app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

    # LAYOUT STRUCTURE
    app.layout = html.Div([
        html.H1('Azerbaijan Statistics 2011-2025', style = {'textAlign':'center'}),

        # Tabs
        dcc.Tabs([
            
            # Demographics Tab
            dcc.Tab(label ='Demographics', 
                children = [
                    html.Div([

                        html.Div([
                            dcc.Dropdown(
                                id = 'year-dropdown-demo',
                                options = [{'label': y, 'value':y} for y in years],
                                value = 2025,
                                clearable = False,
                                style = {'width': '30%'}),

                        dbc.Card(
                            dbc.CardBody([
                                html.H6('Population Growth'),
                                html.H3(id= 'population-kpi')]),
                                className = 'shadow-sm ',
                                style = {'width': '30%'}),
        
                        dbc.Card(
                            dbc.CardBody([
                                html.H6('Natural growth per 1000'),
                                html.H3(id = 'growth-kpi')]),
                                className = 'shadow-sm',
                                style = {'width': '30%'})],
                        style={'display': 'flex','gap': '30px'})],
                        ),

                    
                    html.Div([
                    dcc.Graph(id = 'demographics-column', style = {'width': '50%'}),
                    dcc.Graph(id = 'birth-death-bar',style = {'width': '50%'})],
                    style = {'display': 'flex', 'gap': '20px'}),



                    dcc.Graph(id = 'population-line')]),
                    
                    
            # Economics Tab
            dcc.Tab(label = 'Economics', children = [
                html.Div([

                    html.Div([
                         dcc.Dropdown(
                        id = 'year-dropdown-eco',
                        options = [{'label': y, 'value':y} for y in years],
                        value = 2025,
                        clearable = False,
                        style = {'width': '20%'}),

                        dbc.Card(
                            dbc.CardBody([
                                html.H6('GDP'),
                                html.H3(id = 'GDP-kpi')]),
                                className = 'shadow-sm',
                                style={"width": "26%"}),

                        dbc.Card(
                            dbc.CardBody([
                                html.H6('Inflation'),
                                html.H3(id = 'inflation-kpi')]),
                                className = 'shadow-sm',
                                style={"width": "26%"}),

                        dbc.Card(
                              dbc.CardBody([
                                    html.H6('Current Account'),
                                    html.H3(id ='current-kpi')]),
                                    className = 'shadow-sm',
                                    style={"width": "26%"}),],
                                    style={'display': 'flex','gap': '30px'}),
                        
                    html.Div([
                        dcc.Graph(id = 'export-import-bar',style = {'width': '50%'}),
                        dcc.Graph(id = 'budget-bar',style = {'width': '50%'})],
                        style = {'display': 'flex', 'gap': '20px'}),
                    
                        dcc.Graph(id = 'GDP-line')
                ])])])])

    # CALLBACK STRUCTURE
    @app.callback(
        Output('population-kpi', 'children'),
        Output('growth-kpi', 'children'),
        Output('GDP-kpi', 'children'),
        Output('inflation-kpi', 'children'),
        Output('current-kpi', 'children'),
        Output('demographics-column', 'figure'),
        Output('birth-death-bar', 'figure'),
        Output('population-line', 'figure'),
        Output('export-import-bar', 'figure'),
        Output('budget-bar', 'figure'),
        Output('GDP-line', 'figure'),
        Input('year-dropdown-demo', 'value'),
        Input('year-dropdown-eco', 'value'))

    def update_dashboard(year_demo, year_eco):
        filter_pop = population[population['Year']== year_demo]
        filter_eco = macroeconomics[macroeconomics['Year']== year_eco]
        pop_bar = filter_pop.melt(id_vars = 'Year', value_vars = ['Birth', 'Death'], var_name = 'Type', value_name = 'Rate')
        export_import = filter_eco.melt(id_vars = 'Year', value_vars = ['Export (bln USD)', 'Import (bln USD)'], 
                                        var_name = 'Type', value_name = 'Rate')
        budget = filter_eco.melt(id_vars = 'Year', value_vars = ['Budget revenue', 'Budget expenditure'],
                                var_name = 'Type', value_name = 'Rate')

        #KPI
        pop_growth = filter_pop['Growth change'].iloc[0]
        pop_growth = round(pop_growth*100,4)
        pop_growth = f"{pop_growth:.2f}%"
        growth = filter_pop['Natural growth'].iloc[0]
        GDP = filter_eco['GDP (bln AZN)'].iloc[0]
        GDP = f"{GDP:.1f} bln AZN"
        Inflation = filter_eco['Inflation'].iloc[0]
        Inflation = round(Inflation*100,1)
        Inflation = f"{Inflation:.1f}%"
        Current = filter_eco['Current Account'].iloc[0]
        Current = f"{Current:.1f} bln USD"

        #Charts
        demo_bar = px.bar(demographics, x='Population', y='Age', title='Population share by age in 2025', orientation= 'h')
        birth_death_bar = px.bar(pop_bar, x = 'Type', y = 'Rate', color = 'Type', title = 'Birth and Death statistics per 1000')
        population_line = px.line(population, x='Year', y='Population', title = 'Population by years (1000)')
        export_import_bar = px.bar(export_import, x = 'Type', y='Rate', color = 'Type',title = 'Export and Import')
        budget_bar = px.bar(budget, x='Type', y='Rate', color = 'Type',title = 'Budget revenue and expenditure')
        gdp_line = px.line(macroeconomics,x='Year', y='GDP (bln AZN)', title = 'GDP by year')

        




        
        return pop_growth, growth, GDP, Inflation, Current, demo_bar,birth_death_bar,population_line,export_import_bar, budget_bar, gdp_line
    if __name__ == "__main__":
        webbrowser.open("http://127.0.0.1:8050/")
        app.run(debug = True)